# 📝 에이전트 품질 과제 LV1(기초)

> 이 단원의 새 기술을 **하나씩** 확인합니다: 비평 스키마 만들기, 비평 체인 실행, 토큰 집계, 프롬프트 버전 관리, API 예외 처리.

## 풀이 방법
1. 맨 위 **준비 셀**을 먼저 실행하세요(본인 `OPENAI_API_KEY` 가 든 `.env` 가 필요합니다).
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: `data/gym_members.csv`(헬스장 회원 기록). 이 단원 예시의 배경입니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우
load_dotenv("../../.env") # 교안 폴더 안의 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델 - 실행만 하세요(LangChain 기본 단원에서 만든 것과 같습니다).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('모델 준비 완료:', type(model).__name__)

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 문제의 배경이 되는 헬스장 회원 데이터를 먼저 훑어봅니다. 특히 **회원권별 월방문횟수**는 3번 문제에서 평가할 리포트가 인용하는 수치이니, 그 값을 눈으로 확인해 두세요.

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다
import pandas as pd
gym = pd.read_csv('data/gym_members.csv')
print('회원 수:', len(gym))
print(gym.head())
gym.info()

# 3번 문제의 리포트가 인용하는 수치 - 리포트가 데이터를 제대로 옮겼는지 대조할 근거다
print()
print(gym.groupby('회원권')['월방문횟수'].mean().round(1))

## 1. 비평 스키마 만들기
**배경**: 모델의 비평을 프로그램이 다루려면 **정해진 구조**로 받아야 합니다. 비평 결과를 담을 스키마를 만듭니다.

**요구사항**:
- `BaseModel` 을 상속한 **`ReportCritique`** 클래스를 만드세요.
- 필드는 **`score`**(정수, 1~10 점수)와 **`issues`**(문자열 리스트, 개선점)입니다.
- `score` 에는 `Field(ge=1, le=10)`, `issues` 에는 설명을 붙이세요.

**예시**: `ReportCritique(score=7, issues=['수치 부족'])` 가 만들어지고 `.score` 가 7 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- pydantic BaseModel 을 상속해 필드 두 개를 선언한다.

세부구현:
1. 답안 셀 맨 위에서 from pydantic import BaseModel, Field 로 부품을 가져온다.
2. class ReportCritique(BaseModel): 아래에 score(int)와 issues(list[str])를 선언한다.
3. score 에 Field(ge=1, le=10, ...), issues 에 Field(description=...) 를 붙인다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 'score' in ReportCritique.model_fields and 'issues' in ReportCritique.model_fields
sample = ReportCritique(score=7, issues=['수치 부족'])
assert sample.score == 7 and isinstance(sample.issues, list)
print('✅ 통과!')

## 2. 다른 모양의 스키마 만들기
**배경**: 스키마의 필드는 **목적에 맞게** 자유롭게 정합니다. 이번엔 합격 여부까지 담는 스키마를 만듭니다.

**요구사항**:
- `BaseModel` 을 상속한 **`ReviewGrade`** 클래스를 만드세요.
- 필드는 **`strengths`**(문자열 리스트), **`weaknesses`**(문자열 리스트), **`passed`**(참/거짓, bool)입니다.

**예시**: `ReviewGrade(strengths=['명확'], weaknesses=[], passed=True)` 의 `.passed` 는 True 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1번과 같은 방식으로 필드 세 개(리스트 둘, 불리언 하나)를 선언한다.

세부구현:
1. class ReviewGrade(BaseModel): 를 만든다.
2. strengths: list[str], weaknesses: list[str], passed: bool 을 선언한다.
3. 필요하면 각 필드에 Field(description=...) 를 붙인다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
for field in ('strengths', 'weaknesses', 'passed'):
    assert field in ReviewGrade.model_fields
g = ReviewGrade(strengths=['명확'], weaknesses=[], passed=True)
assert g.passed is True
print('✅ 통과!')

## 3. 비평 체인 실행: 좋은 리포트
**배경**: 1번에서 만든 `ReportCritique` 로 실제 리포트를 평가합니다(구조화된 출력).

**요구사항**:
- 아래 `report_text` 를 `model.with_structured_output(ReportCritique)` 로 평가해 결과를 **`result1`** 에 담으세요.
- 감싼 비평용 모델은 변수 **`critic_model`** 에 담아 두세요(4번에서 그대로 다시 씁니다).
- system 메시지에 `'리포트를 1~10점으로 평가하고 개선점을 지적하라.'`, user 메시지에 `report_text` 를 넣으세요.

**예시**: `result1.score` 는 1~10 사이 정수, `result1.issues` 는 리스트입니다.

<details><summary>힌트</summary>

```text
접근방법:
- with_structured_output(ReportCritique) 로 모델을 감싸 invoke 한다.

세부구현:
1. model.with_structured_output(ReportCritique) 로 비평용 모델을 만들어 critic_model 에 담는다.
2. system·user 두 메시지를 리스트로 invoke 한다(user 에 report_text).
3. 반환 객체를 result1 에 담는다.
```

</details>

In [ ]:
# 평가 대상 1 - 수치·해석·제안이 모두 들어 있는 리포트(위에서 본 15.2 / 10.2 가 그대로 인용돼 있다)
report_text = '12개월 회원의 월 평균 방문은 15.2회로 1개월 회원(10.2회)보다 50% 높다. 장기 회원이 더 자주 방문하므로, 재등록 유도를 위해 장기권 혜택을 강화할 것을 제안한다.'
print(report_text)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(result1, ReportCritique), 'result1 에는 비평 체인이 돌려준 객체를 그대로 담으세요'
assert isinstance(result1.score, int) and 1 <= result1.score <= 10
assert isinstance(result1.issues, list)
# 점수·개선점은 손으로 적어도 위 검사를 통과합니다 - 그래서 비평용 모델을 여기서 한 번 더 직접 불러 봅니다.
probe = critic_model.invoke(
    [{'role': 'system', 'content': '리포트를 1~10점으로 평가하고 개선점을 지적하라.'},
     {'role': 'user', 'content': '방문 횟수가 늘었다.'}])
assert isinstance(probe, ReportCritique) and 1 <= probe.score <= 10
print('✅ 통과!')

## 4. 비평 체인 실행: 부실한 리포트
**배경**: 같은 비평을 **부실한 리포트**에 적용합니다(같은 개념, 다른 입력).

**요구사항**:
- 아래 `weak_text` 를 3번에서 만든 **`critic_model`** 로 평가해 결과를 **`result2`** 에 담으세요.
- 개선점(`issues`)이 몇 개나 나왔는지 세어 보세요.

**예시**: 부실한 리포트라 점수는 낮게, 개선점은 여러 개 나오는 것이 보통입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3번 코드에서 입력 문자열만 weak_text 로 바꾼다.

세부구현:
1. 3번의 critic_model 로 weak_text 를 평가한다(모델을 다시 감쌀 필요가 없다).
2. 결과를 result2 에 담는다.
```

</details>

In [ ]:
# 평가 대상 2 - 같은 데이터를 두고 쓴 글인데 수치도 제안도 없다. 3번과 점수를 견줘 볼 것
weak_text = '회원들이 헬스장을 이용한다. 방문 횟수는 다양하다. 만족도도 괜찮은 편이다.'
print(weak_text)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(result2, ReportCritique), 'result2 에는 비평 체인이 돌려준 객체를 그대로 담으세요'
assert result2 is not result1, '3번 결과를 그대로 두지 말고 weak_text 로 다시 평가하세요'
assert isinstance(result2.score, int) and 1 <= result2.score <= 10
assert isinstance(result2.issues, list)
# 위 검사는 ReportCritique 를 손으로 만들어 담아도 통과합니다 - 3번처럼 비평용 모델을
# 여기서 한 번 더 직접 불러, weak_text 를 실제로 평가할 수 있는 상태인지 확인합니다.
probe2 = critic_model.invoke(
    [{'role': 'system', 'content': '리포트를 1~10점으로 평가하고 개선점을 지적하라.'},
     {'role': 'user', 'content': weak_text}])
assert isinstance(probe2, ReportCritique) and 1 <= probe2.score <= 10
print('✅ 통과!')

## 5. 리포트 수정하기
**배경**: 비평의 개선점을 반영해 리포트를 **다시 쓰는** 함수를 만듭니다(성찰 루프의 '수정' 단계).

**요구사항**:
- 함수 **`revise_report(report, issues)`** 를 만드세요. `report`(원본 문자열)와 `issues`(개선점 리스트)를 받아 모델에게 **개선점을 반영해 다시 쓰게** 하고, 수정된 **문자열**을 돌려줍니다.
- system 에 `'리포트를 개선점을 반영해 다시 써라.'`, user 에 원본과 개선점을 함께 담으세요.

**예시**: `revise_report('회원 수는 160명이다.', ['재등록률 70% 를 문장에 추가할 것'])` 는 **원본에 없던 70 이 들어간** 새 문장을 돌려줍니다.

> 자가채점은 돌려준 글이 **원본과 다른지**, 그리고 개선점으로 요구한 **70 이 들어 있는지**를 봅니다. 원본을 그대로 돌려주는 함수는 통과하지 못합니다.

<details><summary>힌트</summary>

```text
접근방법:
- user 문자열에 원본 리포트와 개선점 목록을 함께 넣어 invoke 한다.

세부구현:
1. issues 를 줄바꿈으로 이어 하나의 문자열로 만든다.
2. system·user 메시지로 model.invoke(...) 를 호출한다.
3. 응답의 .text 를 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 문자열을 돌려주기만 하면 통과하는 검사는, 모델을 안 부르고 아무 글자나 반환해도 통과합니다.
# 원본에 이미 있는 값을 요구하면 원본을 그대로 돌려줘도 통과하므로, 원본에 '없던' 값을 요구합니다.
source = '회원 수는 160명이다.'
out = revise_report(source, ['재등록률 70% 를 문장에 추가할 것'])
assert isinstance(out, str) and len(out.strip()) > 0
assert out.strip() != source, '원본을 그대로 돌려주면 개선점을 반영해 다시 쓴 것이 아닙니다'
assert '70' in out, '개선점으로 요구한 재등록률 70 이 다시 쓴 글에 들어 있어야 합니다'
print('✅ 통과!')

## 6. 토큰 사용량 집계
**배경**: 여러 번 호출한 총 토큰을 더해 비용을 가늠합니다(관측성의 기본).

**요구사항**:
- 아래 `questions` 의 각 질문을 `model.invoke(...)` 로 호출하고, 응답 `usage_metadata` 의 `'total_tokens'` 를 읽어, 질문 순서대로 리스트 **`token_log`** 에 담고 그 합을 **`total_tokens`** 에 담으세요.
- 값이 없을 때를 대비해 `.get('total_tokens', 0)` 으로 꺼내세요.

**예시**: 질문이 2개면 `token_log` 의 길이는 2 이고, `total_tokens` 는 `sum(token_log)` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문마다 토큰을 리스트에 모아 두고, 마지막에 합을 낸다.

세부구현:
1. token_log = [] 로 시작한다.
2. questions 를 돌며 invoke 한 응답의 usage_metadata.get('total_tokens', 0) 을 token_log 에 append 한다.
3. total_tokens 에 그 리스트의 합을 담는다.
```

</details>

In [ ]:
# 짧은 질문 두 개 - 토큰 집계가 목적이라 답 내용은 중요하지 않다
questions = ['1 더하기 1은?', '무지개는 몇 색?']

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# total_tokens >= 0 만 보면 0 을 손으로 적어도 통과합니다 - 호출별 기록과 합계를 함께 봅니다.
assert isinstance(token_log, list) and len(token_log) == len(questions)
assert all(isinstance(t, int) and t > 0 for t in token_log), '실제 호출이라면 질문마다 토큰이 0 보다 큽니다'
assert isinstance(total_tokens, int) and total_tokens == sum(token_log)
print('✅ 통과!')

## 7. 프롬프트 버전 꺼내기
**배경**: 프롬프트를 코드 밖 사전에 **버전별**로 두고 이름·버전으로 꺼냅니다.

**요구사항**:
- 함수 **`get_version(registry, name, version)`** 를 만드세요. 중첩 사전에서 `registry[name][version]` 문자열을 돌려줍니다.

**예시**: 아래 `reg` 로 `get_version(reg, 'writer', 'v1')` 은 `'v1 프롬프트'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 중첩 사전을 두 번 인덱싱해 돌려준다.

세부구현:
1. 인자 세 개를 받는 함수를 정의한다.
2. 바깥 사전에서 이름으로 한 번, 그 안에서 버전으로 한 번, 이렇게 두 단계로 찾아 돌려준다.
```

</details>

In [ ]:
# 이름 아래 버전이 들어 있는 2단 구조 - 실제 레지스트리(data/prompts.yaml)와 같은 모양이다
reg = {'writer': {'v1': 'v1 프롬프트', 'v2': 'v2 프롬프트'},
       'critic': {'v1': '평가 v1'}}

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert get_version(reg, 'writer', 'v1') == 'v1 프롬프트'
assert get_version(reg, 'critic', 'v1') == '평가 v1'
# 위 두 줄은 답을 함수 안에 적어 둬도 통과합니다 - 처음 보는 사전으로 한 번 더 확인합니다.
probe_reg = {'writer': {'v1': '첫 판', 'v9': '아홉째 판'}}
assert get_version(probe_reg, 'writer', 'v9') == '아홉째 판', '넘겨받은 사전에서 찾아야 합니다'
print('✅ 통과!')

## 8. 프롬프트 새 버전 추가하기
**배경**: 프롬프트를 개선하면 **새 버전**으로 추가합니다(기존 버전은 보존).

**요구사항**:
- 함수 **`add_version(registry, name, text)`** 를 만드세요. `registry[name]` 에 다음 번호 버전(`v` + 다음 숫자)을 키로 `text` 를 추가하고, **추가한 버전 문자열**(예: `'v3'`)을 돌려줍니다.
- 다음 번호는 **현재 버전 개수 + 1** 로 정하세요(v1·v2 가 있으면 새 버전은 v3).

**예시**: `reg['writer']` 에 v1·v2 가 있으면 `add_version(reg, 'writer', '새 프롬프트')` 는 `'v3'` 을 돌려주고 `reg['writer']['v3']` 가 생깁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 현재 버전 개수로 다음 번호를 정해 키를 만든다.

세부구현:
1. 다음 번호 = len(registry[name]) + 1 로 계산한다.
2. 새 키 문자열 'v{다음번호}' 를 만든다.
3. registry[name][새키] = text 로 넣고 새 키를 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
new_label = add_version(reg, 'critic', '평가 개선본')
assert new_label == 'v2'                 # critic 은 v1 만 있었으니 새 버전은 v2
assert reg['critic']['v2'] == '평가 개선본'

# 버전이 이미 셋 있는 레지스트리로 한 번 더 - '다음 번호 = 개수 + 1' 규칙을 구현했는지 봅니다
# ('v2' 처럼 번호를 고정해 두면 여기서 걸립니다).
probe_reg = {'writer': {'v1': '첫 판', 'v2': '둘째 판', 'v3': '셋째 판'}}
assert add_version(probe_reg, 'writer', '넷째 판') == 'v4', '다음 번호는 현재 버전 개수 + 1 입니다'
assert probe_reg['writer']['v4'] == '넷째 판'
assert len(probe_reg['writer']) == 4, '기존 버전을 덮어쓰면 안 됩니다'
print('✅ 통과!')

## 9. API 호출 예외 처리
**배경**: 실제 서비스에서 호출은 실패할 수 있습니다. 실패해도 **프로그램이 멈추지 않게** 예외를 붙잡아 **문자열로 보고**합니다.

**요구사항**:
- 함수 **`safe_call(func)`** 를 만드세요. `func()` 를 `try`/`except` 로 감싸, 성공하면 그 반환값을, **실패하면 `'실패: ' + 에러메시지`** 문자열을 돌려줍니다.

**예시**: 아래 `broken()` 처럼 예외를 던지는 함수를 넣으면 `'실패: ...'` 로 시작하는 문자열이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- try 에서 func() 를 반환하고, except 에서 실패 문자열을 반환한다.

세부구현:
1. 함수를 인자로 받아, try 블록 안에서 그 함수를 실행해 결과를 그대로 돌려준다.
2. except 로 예외 객체를 붙잡아, 요구사항에 적힌 접두사에 에러 메시지를 이어 붙인 문자열을 돌려준다.
```

</details>

In [ ]:
# 실패를 우리가 정한 시점에 일으키는 함수 - 진짜 장애를 기다리지 않고 예외 처리를 확인한다
def broken():
    raise RuntimeError('모델 이름이 잘못되었습니다')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
out = safe_call(broken)
assert isinstance(out, str) and out.startswith('실패:')
# 접두사만 붙이고 끝내면 무엇이 실패했는지 알 수 없습니다 - 에러 메시지까지 들어가야 합니다.
assert '모델 이름이 잘못되었습니다' in out, '실패 문자열에 에러 메시지를 이어 붙이세요'
assert safe_call(lambda: '정상 응답') == '정상 응답'
print('✅ 통과!')

## 10. 관측성 용어 정리 (서술형)
**배경**: 관측 도구(Langfuse)는 실행을 **trace·span·generation** 세 단위로 기록합니다.

**요구사항**: 아래 markdown 셀에 세 용어의 **포함 관계**(무엇이 무엇을 담는지)와 각 용어의 뜻을 **자신의 말로** 설명하세요. 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 자신의 설명을 서술하세요: trace·span·generation 의 포함 관계와 뜻)*

---
수고했어요! LV1 에서 비평 스키마·비평 체인·토큰 집계·프롬프트 버전·예외 처리를 **하나씩** 익혔습니다. LV2 에서는 이것들을 **조합**해 성찰 루프와 견고한 호출을 만듭니다.